<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/MobilNet%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cargar Base

In [1]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path1 = kagglehub.dataset_download("leonardocaravaggio/ge-images")
path2 = kagglehub.dataset_download("leonardocaravaggio/ge-images2")
path3 = kagglehub.dataset_download("leonardocaravaggio/ge-images3")

100%|██████████| 13.3G/13.3G [02:48<00:00, 84.6MB/s]

Extracting files...


100%|██████████| 13.9G/13.9G [02:40<00:00, 93.4MB/s]

Extracting files...


100%|██████████| 162M/162M [00:01<00:00, 114MB/s]

Extracting files...


In [2]:
import os
import shutil

# Crear una carpeta de destino
dest_folder = "imagenes"
os.makedirs(dest_folder, exist_ok=True)

# Función para copiar imágenes a una sola carpeta
def mover_imagenes(origen, destino):
    for root, _, files in os.walk(origen):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                shutil.move(os.path.join(root, file), os.path.join(destino, file))

# Copiar imágenes de ambos datasets al mismo folder
mover_imagenes(path1, dest_folder)
mover_imagenes(path2, dest_folder)

print(f"Imágenes combinadas en la carpeta: {dest_folder}")

Imágenes combinadas en la carpeta: imagenes


In [3]:
import pandas as pd
ciudades=pd.read_csv("base.csv")

In [4]:
len(ciudades)

1095

In [17]:
ciudades['Desigualdad_10km']=np.nan
ciudades['Desigualdad_1km']=np.nan
ciudades['Diferencia']=np.nan

#MobilNet v2

In [32]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import os

# Cargar MobileNetV2 hasta la capa 12
model = models.mobilenet_v2(pretrained=True)
model = torch.nn.Sequential(*list(model.features[:12]))
model.eval()

# Transformaciones de imagen
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Función para extraer características
def extract_index(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)  # [1, 3, 224, 224]

    with torch.no_grad():
        features = model(image_tensor)  # Pasar por la red
        avg_pool = F.adaptive_avg_pool2d(features, (1, 1)).squeeze()  # [C]
        max_pool = F.adaptive_max_pool2d(features, (1, 1)).squeeze()  # [C]
        pooled_features = torch.cat((avg_pool, max_pool), dim=0)  # [2C]
        inequality_index = np.std(pooled_features.cpu().numpy())  # desigualdad

    return inequality_index

# Función para computar desigualdad a múltiples escalas
def compute_inequality(image_1km_path, image_5km_path, image_10km_path, image_15km_path):
    return {
        "Desigualdad_1km": extract_index(image_1km_path),
        "Desigualdad_5km": extract_index(image_5km_path),
        "Desigualdad_10km": extract_index(image_10km_path),
        "Desigualdad_15km": extract_index(image_15km_path),
    }

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [34]:
import os

lista_ciudades = ["Rocinha", "Retiro", "Oceano", "Amazonas", "Lo Barnechea", "El Alto", "Desierto"]

for nombre in lista_ciudades:

    nombre_archivo = nombre.replace("/", ".").replace(":", "_").replace("'", "!")
    ruta_completa = os.path.join(path3, nombre_archivo)

    # Armar paths a las imágenes
    img_1k = ruta_completa + " - 1K.png"
    img_5k = ruta_completa + " - 5K.png"
    img_10k = ruta_completa + " - 10K.png"
    img_15k = ruta_completa + " - 15K.png"

    # Llamar a la función
    print(nombre, compute_inequality(img_1k, img_5k, img_10k, img_15k))


Rocinha {'Desigualdad_1km': np.float32(0.46365294), 'Desigualdad_5km': np.float32(0.497338), 'Desigualdad_10km': np.float32(0.4949383), 'Desigualdad_15km': np.float32(0.56390446)}
Retiro {'Desigualdad_1km': np.float32(0.549047), 'Desigualdad_5km': np.float32(0.52183765), 'Desigualdad_10km': np.float32(0.47088647), 'Desigualdad_15km': np.float32(0.5221058)}
Oceano {'Desigualdad_1km': np.float32(0.46293652), 'Desigualdad_5km': np.float32(0.46268353), 'Desigualdad_10km': np.float32(0.4628366), 'Desigualdad_15km': np.float32(0.45829245)}
Amazonas {'Desigualdad_1km': np.float32(0.3445576), 'Desigualdad_5km': np.float32(0.38959622), 'Desigualdad_10km': np.float32(0.4056827), 'Desigualdad_15km': np.float32(0.44981283)}
Lo Barnechea {'Desigualdad_1km': np.float32(0.49379194), 'Desigualdad_5km': np.float32(0.47963837), 'Desigualdad_10km': np.float32(0.48359868), 'Desigualdad_15km': np.float32(0.4766983)}
El Alto {'Desigualdad_1km': np.float32(0.47399375), 'Desigualdad_5km': np.float32(0.4705086

In [35]:
# Procesamiento por ciudad
for i in range(1095):
    if pd.isna(ciudades.loc[i, "Diferencia"]):
        try:
            path='/content/'
            nombre_archivo = ciudades.City[i].replace("/", ".").replace(":", "_").replace("'", "!")
            ruta_completa = os.path.join(path, "imagenes", nombre_archivo)


            img_1k = ruta_completa + " - 1K.png"
            img_5k = ruta_completa + " - 5K.png"
            img_10k = ruta_completa + " - 10K.png"
            img_15k = ruta_completa + " - 15K.png"

            if not os.path.exists(img_1k):
                print(f"❌ No existe: {img_1k}")
                continue

            if not os.path.exists(img_5k):
                print(f"❌ No existe: {img_5k}")
                continue

            if not os.path.exists(img_10k):
                print(f"❌ No existe: {img_10k}")
                continue

            if not os.path.exists(img_15k):
                print(f"❌ No existe: {img_15k}")
                continue

            resultados = compute_inequality(img_1k, img_5k, img_10k, img_15k)
            ciudades.loc[i, "Desigualdad_1km"] = resultados["Desigualdad_1km"]
            ciudades.loc[i, "Desigualdad_5km"] = resultados["Desigualdad_5km"]
            ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad_10km"]
            ciudades.loc[i, "Desigualdad_15km"] = resultados["Desigualdad_15km"]
            #ciudades.loc[i, "Diferencia"] = resultados["Diferencia"]

        except Exception as e:
            print(f"⚠️ Error en {ciudades.City[i]}: {e}")


# Bajar la base con el indicador de desigualdad

In [36]:
import statsmodels.api as sm
import numpy as np

# Definir variables
X = ciudades["Desigualdad_10km"].replace([np.inf, -np.inf], np.nan)
y = ciudades["P1ST"].replace([np.inf, -np.inf], np.nan)

# Filtrar filas con NaN en X o y
mask = X.notna() & y.notna()
X, y = X[mask], y[mask]

# Agregar constante para la ordenada al origen
X = sm.add_constant(X)

# Ajustar modelo
modelo = sm.OLS(y, X).fit()

# Resumen de la regresión
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   P1ST   R-squared:                       0.059
Model:                            OLS   Adj. R-squared:                  0.058
Method:                 Least Squares   F-statistic:                     68.83
Date:                Tue, 22 Apr 2025   Prob (F-statistic):           3.14e-16
Time:                        16:57:09   Log-Likelihood:                -454.57
No. Observations:                1095   AIC:                             913.1
Df Residuals:                    1093   BIC:                             923.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                4.1786      0.129  

In [37]:
from google.colab import files
name="base_mobilv2.csv"
ciudades.to_csv(name)
files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>